# 🧠 Дообучение Толяна (`saiga-llama3:8b`) под вашу конфу в Дискорде

**Как запустить (100% бесплатно в Google Colab):**
1. В верхнем меню выбери **Среда выполнения (Runtime) → Сменить среду выполнения (Change runtime type) → T4 GPU**.
2. Слева нажми на иконку папки 📁 и перетащи туда файл **`dataset.jsonl`** из папки `finetune/`.
3. Нажми **Среда выполнения (Runtime) → Выполнить все (Run all)**.
4. Через ~12–15 минут скачается готовый `.gguf` файл модели (или сохранится на твой Google Диск) — перекинь его на ноутбук с Linux и выполни:
   ```bash
   ollama create tolyan -f finetune/Modelfile
   ```

In [ ]:
# 1. Установка Unsloth (быстрое обучение на GPU)
!pip install -q unsloth

In [ ]:
# 2. Загрузка базовой модели Saiga Llama-3 8B в 4-битном режиме
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="IlyaGusev/saiga_llama3_8b",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("✅ Модель загружена и готова к дообучению!")

In [ ]:
# 3. Загрузка датасета вашей компании (dataset.jsonl)
import os
from datasets import load_dataset

if not os.path.exists("dataset.jsonl"):
    from google.colab import files
    print("Загрузи файл dataset.jsonl из папки finetune/:")
    files.upload()

dataset = load_dataset("json", data_files="dataset.jsonl", split="train")

def format_prompts(examples):
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in examples["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)
print(f"✅ Загружено диалогов для обучения: {len(dataset)}")

In [ ]:
# 4. Запуск обучения (занимает ~5-10 минут на T4 GPU)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=4,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
    ),
)

trainer.train()
print("🔥 Толян успешно дообучен на диалогах вашей конфы!")

In [ ]:
# 5. Тестовая проверка обученного Толяна прямо в Colab
FastLanguageModel.for_inference(model)
test_messages = [
    {"role": "user", "content": "Толян, где Ярик опять пропал и куда смок в КС кидать?"}
]
inputs = tokenizer.apply_chat_template(
    test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=110, temperature=0.68, top_p=0.9)
print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))

In [ ]:
# 6. Экспорт в GGUF под 4 ГБ VRAM (GTX 1650) и сохранение на Google Диск
# q4_k_m — стандартное качество (как в оригинальной saiga-llama3:8b)
# q3_k_m — чуть легче по весу (~3.8 ГБ), чтобы целиком помещаться в 4 ГБ видеопамяти GTX 1650!
QUANT_METHOD = "q4_k_m"  # можешь поменять на "q3_k_m" для максимальной скорости на GTX 1650

model.save_pretrained_gguf("tolyan_gguf", tokenizer, quantization_method=QUANT_METHOD)

# По желанию: копируем готовый .gguf сразу на твой Google Диск, чтобы удобно скачать
from google.colab import drive
drive.mount("/content/drive")
!cp tolyan_gguf*/*.gguf /content/drive/MyDrive/tolyan-saiga-llama3-8b.gguf
print("✅ Готово! Файл tolyan-saiga-llama3-8b.gguf сохранён в корень твоего Google Диска!")